In [0]:
# This is just a basic notebook that trains our ML model on the full dataset, with Spark ML.

from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import Imputer, OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler

# We load the predictors
predictors = spark.table("workspace.default.gold_station_predictors")

# We load the targets. The filter is unnecessary at the time of writing because we only have been exploring one horizon
targets = spark.table("workspace.default.gold_station_targets").filter("horizon_minutes = 30")

# We put them together
full_data = predictors.join(targets, ["station_id", "fetched_at"])

# We list the numeric features

numeric_features_bike = ["num_bikes_available", "bike_available_fraction",
               "bike_change_30min_per_minute_fraction", "log_capacity", "weekend",
               "hour_sin_1", "hour_cos_1", "hour_sin_2", "hour_cos_2"]

# Spark does not need the data in memory, so we use all rows (we don't filter to 10% like for sklearn)

data_bike = (full_data.filter(F.col("bike_target").isNotNull()) # Remove null rows
            .select(*numeric_features_bike, "geographic_cell", "bike_target") # Unpack the numerics, select the geographic cells, and the target
        )

# Spark ML pipelines are organized into stages. Usually we 
# would name the output of each stage during the completion of that stage, but
# for the imputed columns there is a name for each column so we do it first
# We could alternatively put this in the Imputer line
imputed_bike = [f"{c}_imputed" for c in numeric_features_bike]

# SparkML penalties are different. C = 0.25 in sklearn is the same as regParam = 1 / (C * number of rows) in SparkML
# It has to do with how and where the penalties are applied, in sklearn the penalty "C" is like the scale of a prior.
# So "C=0" is "We believe all the coefficients = 0". In SparkML it is more usual. Also, in sklearn the penalty
# term is an average, in SparkML it is a sum, so we have to additionally divide by the number of data points to 
# make them equivalent.

reg_param_bike = 1 / (0.25 * data_bike.count())

# Here is our Spark ML pipeline

pipeline_bike = Pipeline(stages=[
    Imputer(strategy="median", inputCols=numeric_features_bike, outputCols=imputed_bike), # Median imputation
    VectorAssembler(inputCols=imputed_bike, outputCol="num_vector"), # Vector assembler smooshes the numeric features together 
    StandardScaler(inputCol="num_vector", outputCol="num_scaled", withMean=True, withStd=True), # Standardizes the numeric features
    StringIndexer(inputCol="geographic_cell", outputCol="cell_index", handleInvalid="keep"), # Converts the geographic
    # cell names to numbers
    OneHotEncoder(inputCol="cell_index", outputCol="cell_onehot"), # One-Hot Encoding the geographic cells. The default dropLast removes the unknown slot, so unknown cells get all zeros
    VectorAssembler(inputCols=["num_scaled", "cell_onehot"], outputCol="features"), # Spark models read one vector column
    LogisticRegression(featuresCol="features", labelCol="bike_target",
                       regParam=reg_param_bike, maxIter=200, standardization=False), # Logistic regression
    ])

model_bike = pipeline_bike.fit(data_bike)